# v1 sanity check: average 3PM by player (2025-26)

This notebook is a lightweight research sandbox artifact for `00_research`.

Goal:
- Pull 2025-26 player game logs from S3 via DuckDB
- Compute average `FG3M` by player
- Sort descending to confirm expected leaderboard behavior

In [13]:
import os, subprocess

# pull creds from your AWS CLI config
os.environ["AWS_ACCESS_KEY_ID"] = subprocess.check_output(
    ["aws", "configure", "get", "aws_access_key_id"], text=True
).strip()

os.environ["AWS_SECRET_ACCESS_KEY"] = subprocess.check_output(
    ["aws", "configure", "get", "aws_secret_access_key"], text=True
).strip()

os.environ["AWS_DEFAULT_REGION"] = "us-east-2"
# Optional: some libs look for AWS_REGION too
os.environ["AWS_REGION"] = "us-east-2"

for k in ["AWS_ACCESS_KEY_ID","AWS_SECRET_ACCESS_KEY","AWS_DEFAULT_REGION","AWS_REGION","AWS_SESSION_TOKEN"]:
    print(k, "=>", "SET" if k in os.environ else "MISSING")

AWS_ACCESS_KEY_ID => SET
AWS_SECRET_ACCESS_KEY => SET
AWS_DEFAULT_REGION => SET
AWS_REGION => SET
AWS_SESSION_TOKEN => MISSING


In [14]:
import os
import duckdb
import pandas as pd

con = duckdb.connect()
con.execute("INSTALL httpfs")
con.execute("LOAD httpfs")
con.execute("SET s3_region='us-east-2'")
con.execute(f"SET s3_access_key_id='{os.environ['AWS_ACCESS_KEY_ID']}'")
con.execute(f"SET s3_secret_access_key='{os.environ['AWS_SECRET_ACCESS_KEY']}'")
if 'AWS_SESSION_TOKEN' in os.environ:
    con.execute(f"SET s3_session_token='{os.environ['AWS_SESSION_TOKEN']}'")

query = """
SELECT
  PLAYER_NAME,
  AVG(FG3M) AS avg_3pm,
  COUNT(*) AS games
FROM read_csv_auto('s3://nba-api-mt/player_game_logs/2025-26/*.csv', union_by_name=true)
GROUP BY PLAYER_NAME
HAVING COUNT(*) >= 5
ORDER BY avg_3pm DESC
LIMIT 25
"""

leaders_df = con.execute(query).fetchdf()
leaders_df


,PLAYER_NAME,avg_3pm,games
0,Stephen Curry,4.487179,39
1,Luka Doncic,3.800000,5
2,Luka Dončić,3.659091,44
3,LaMelo Ball,3.480769,52
4,Kon Knueppel,3.475410,61
5,Donovan Mitchell,3.472727,55
6,Anthony Edwards,3.461538,52
7,Grayson Allen,3.384615,39
8,Michael Porter Jr.,3.367347,49
9,Sam Merrill,3.358974,39


In [16]:
os.getcwd()

'/Users/thomasmyles/dev/betting/src/nba_three_point_modeling/00_research/notebooks'

In [19]:
# 
import sys

SRC_DIR = "/Users/thomasmyles/dev/betting/src"
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

from player_name_utils import normalize_player_name

# from src.player_name_utils import normalize_player_name

query = """
SELECT
  PLAYER_NAME,
  FG3M
FROM read_csv_auto('s3://nba-api-mt/player_game_logs/2025-26/*.csv', union_by_name=true)
"""

raw_df = con.execute(query).fetchdf()

# Normalize names first so Luka Dončić + Luka Doncic collapse to one player
raw_df["PLAYER_NAME"] = raw_df["PLAYER_NAME"].apply(normalize_player_name)

leaders_df = (
    raw_df.groupby("PLAYER_NAME", as_index=False)
    .agg(
        avg_3pm=("FG3M", "mean"),
        games=("FG3M", "size"),
    )
    .query("games >= 5")
    .sort_values("avg_3pm", ascending=False)
    .head(25)
)

leaders_df

,PLAYER_NAME,avg_3pm,games
480,Stephen Curry,4.487179,39
351,Luka Doncic,3.673469,49
338,Lamelo Ball,3.480769,52
328,Kon Knueppel,3.475410,61
141,Donovan Mitchell,3.472727,55
27,Anthony Edwards,3.461538,52
178,Grayson Allen,3.384615,39
375,Michael Porter Jr,3.367347,49
464,Sam Merrill,3.358974,39
518,Tyrese Maxey,3.338983,59
